# 00. pytest

**Tier:** Intermediate. **Why you'd use this:** A `word_count` helper must stay correct when you change the splitter. A `test_` function with `assert` fails loudly when the count drifts.

**After this lesson you can:** write `test_` functions; read a pytest failure; use a fixture like `tmp_path` for file isolation.
About 25 minutes. **Uses:** [Definitions](../05-functions/01-definitions-lesson.ipynb), [Paths](../14-files/01-paths-lesson.ipynb)


## Quick review

1. Why prefer aware datetimes for instants?
2. What does `encoding="utf-8"` protect you from?

<details><summary>Answers</summary>

Naive values have no zone. Platform-dependent text decoding.
</details>


## The idea

pytest collects callables named `test_*` and runs them. A false `assert` fails the test. pytest rewrites asserts so failures show the values. Fixtures such as `tmp_path` hand each test a fresh temporary directory so file tests do not share state.

<details><summary>Explain more</summary>

`assert word_count("a b") == 2` either passes silently or fails with both sides printed. Isolation means one test's files do not break the next.
</details>


## See it work

Assert a word count, then fail one on purpose under pytest.


In [1]:
def word_count(text: str) -> int:
    # 1. split() with no argument drops empty gaps.
    return len(text.split())


def test_word_count() -> None:
    assert word_count("") == 0
    assert word_count("one two") == 2


test_word_count()
print("passed")


passed


## Check yourself

Answer these before you run the cell.

1. What does pytest show when an assert fails that a bare assert traceback often hides?
2. Why give each test its own `tmp_path`?


In [2]:
import subprocess
import sys
from pathlib import Path

sample = Path("exercise/_fail_demo.py")
sample.write_text(
    "def test_fail():\n    assert 1 + 1 == 3\n",
    encoding="utf-8",
)
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(sample)],
    capture_output=True,
    text=True,
)
print(result.stdout)
print(result.returncode)
sample.unlink()


F                                                                        [100%]
=================================== FAILURES ===================================
__________________________________ test_fail ___________________________________

    def test_fail():
>       assert 1 + 1 == 3
E       assert (1 + 1) == 3

exercise/_fail_demo.py:2: AssertionError
=========================== short test summary info ============================
FAILED exercise/_fail_demo.py::test_fail - assert (1 + 1) == 3
1 failed in 0.02s

1


<details><summary>Answers</summary>

The compared values, thanks to assertion rewriting. So tests do not share leftover files.
</details>


## Why it works this way

A test is a cheap, repeatable claim about behaviour. pytest turns ordinary asserts into rich failures by rewriting the assert bytecode before collection runs. That is why you see both sides of a comparison in the report without writing `self.assertEqual`. Collection discovers tests by name, not by inheritance, which is why plain functions work. Markers and parametrize come later; master asserts and isolation first.

Fixtures are dependency injection for tests. `tmp_path` creates a unique temporary directory per test and cleans it up afterward. That isolation prevents order-dependent failures when two tests write the same filename. Shared mutable globals are the usual way tests become flaky.

Cost of a unit test should stay small: pure functions, no network. The design reason is confidence to change code. Evidence: a passing local assert, and a subprocess pytest run that prints the failed comparison and a non-zero exit code.

Run the exercise tests under `exercise/` when you want a longer suite. Keep test names descriptive. One logical claim per test beats a single mega-assert. Re-run after every fix so you only change what you meant.

Assertion rewriting only applies under pytest collection. Running a test function by hand in a notebook still uses ordinary asserts. That is fine for a quick check, and incomplete as proof of the suite. Prefer `python -m pytest path` when you want the real reporter.

Parametrize comes after you have one clear example. Copy-pasting three near-identical tests is a signal to parametrize, not a virtue. Keep the first version obvious. Fixtures beat setup modules that mutate process-wide state. If two tests need the same file contents, write a fixture that returns a path under `tmp_path` instead of committing a golden file you both mutate.


In [3]:
import subprocess
import sys
from pathlib import Path


def word_count(text: str) -> int:
    return len(text.split())

assert word_count("a b c") == 3
sample = Path("exercise/_pass_demo.py")
sample.write_text(
    "def test_ok():\n    assert 2 + 2 == 4\n",
    encoding="utf-8",
)
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(sample)],
    capture_output=True,
    text=True,
)
print(result.stdout.strip())
print(result.returncode)
sample.unlink()


.                                                                        [100%]
1 passed in 0.01s
0


## When to use it

Use pytest for new Python tests. Use `tmp_path` for file IO. Keep tests independent. Don't rely on test order. Don't leave network or clock calls unmocked when they make failures random. Treat a red suite as blocked work, not as noise to scroll past.


## Common mistake

Writing a test that passes only because a previous test created a file on disk.


In [4]:
from pathlib import Path

shared = Path("exercise/_shared_out.txt")
shared.write_text("stale", encoding="utf-8")
assert shared.read_text(encoding="utf-8") == "stale"
print("looks fine until another test deletes the file")
shared.unlink()


looks fine until another test deletes the file


Take `tmp_path` as a fixture argument in real pytest tests so each run gets a clean directory.


In [5]:
print("def test_write(tmp_path):")
print('    target = tmp_path / "out.txt"')
print('    target.write_text("ok", encoding="utf-8")')
print('    assert target.read_text(encoding="utf-8") == "ok"')


def test_write(tmp_path):
    target = tmp_path / "out.txt"
    target.write_text("ok", encoding="utf-8")
    assert target.read_text(encoding="utf-8") == "ok"


## Try it

Open `exercise/test_checks.py` and run pytest on it. Which failure message shows assertion rewriting most clearly?


## Recap

- pytest runs `test_*` functions and rewrites asserts.
- Failures show values. Non-zero exit means red.
- Fixtures like `tmp_path` keep file tests isolated.

**Next:** [practice](00-pytest-practice.ipynb)
